In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# The most work a fuel can do: methane, gasoline, and a spoonful of sugar (Illustrations 14.5-1, 14.5-2 and 14.5-3)

$$-W_{S,\max}=\sum_i N_{1,i}B_i-\sum_i N_{2,i}B_i
\qquad\qquad B_i=\overline{H}_i-T_{\rm amb}\overline{S}_i$$

Section 14.5 asks the same question of three fuels and gets three different lessons out of it.
Combustion goes to completion, so there is no equilibrium problem at all -- the whole calculation is
bookkeeping on availability, and what changes between the three is *which term you are allowed to
drop*.

| | fuel | what it isolates |
|---|---|---|
| **14.5-1** | methane, in stoichiometric air | that the $RT\ln y$ terms can cancel **exactly**, and why |
| **14.5-2** | *n*-octane, a liquid at ambient | that they do **not** cancel when the stoichiometry is unbalanced |
| **14.5-3** | glucose, at body temperature | that $\Delta_fG$ is only the availability *at 25 °C*, and 37 °C needs $H$ and $S$ separately |

There is no figure in this section. What it produces is **tables** -- three of them -- and this
notebook generates them: the numbers below are the numbers the section prints. Each is pinned with
an assertion so it cannot drift silently.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd

from thermo.data import get_reaction_species

R = 8.314
T_AMB = 298.15
T_BODY = 310.15

def dGf(name):
    """Standard Gibbs energy of formation at 25 C, kJ/mol, from Appendix A.IV."""
    return float(get_reaction_species(name)["DG"])

for s in ("CH4", "O2", "N2", "CO2", "H2O(g)"):
    print(f"  dGf({s:7s}) = {dGf(s):+8.1f} kJ/mol")
AIR = 3.76        # mol N2 per mol O2

  dGf(CH4    ) =    -50.5 kJ/mol
  dGf(O2     ) =     +0.0 kJ/mol
  dGf(N2     ) =     +0.0 kJ/mol
  dGf(CO2    ) =   -394.4 kJ/mol
  dGf(H2O(g) ) =   -228.6 kJ/mol


## Illustration 14.5-1 -- methane

Part [a] drops the $RT\ln y$ terms, so $W_{S,\max}$ is just $\Delta_{\rm rxn}G^{\circ}$. Part [b]
keeps them and gets the same answer, which the illustration calls fortuitous. It is not luck -- it is
structural, and the argument for it is short.

In [3]:
# CH4 + 2 O2 = CO2 + 2 H2O
dG_rxn = dGf("CO2") + 2 * dGf("H2O(g)") - dGf("CH4") - 2 * dGf("O2")
print(f"  [a] W_S,max = dG_rxn = {dG_rxn:+.1f} kJ/mol methane")
assert abs(dG_rxn - (-801.1)) < 0.05

  [a] W_S,max = dG_rxn = -801.1 kJ/mol methane


In [4]:
def N_ln_y(moles):
    """sum_i N_i ln y_i for a gas mixture given as {species: moles}."""
    tot = sum(moles.values())
    return sum(N * np.log(N / tot) for N in moles.values() if N > 0), tot

# Stoichiometric air: 2 x 3.76 = 7.52 mol of nitrogen. The second line perturbs it to
# show that the cancellation below is structural rather than arithmetic -- it survives a
# nitrogen number that is slightly off, which is exactly why an error in that number
# could hide here.
for label, nN2 in (("stoichiometric (7.52)", 2 * AIR), ("perturbed (7.56)", 7.56)):
    inlet = {"CH4": 1.0, "O2": 2.0, "N2": nN2}
    outlet = {"CO2": 1.0, "H2O": 2.0, "N2": nN2}
    si, ti = N_ln_y(inlet)
    so, to = N_ln_y(outlet)
    print(f"  {label:24s} inlet total {ti:6.2f}  sum N ln y = {si:8.4f}")
    print(f"  {'':24s} outlet total {to:6.2f}  sum N ln y = {so:8.4f}"
          f"    difference {so - si:+.2e}")
    assert abs(so - si) < 1e-12          # exact, for either nitrogen number

  stoichiometric (7.52)    inlet total  10.52  sum N ln y =  -8.1981
                           outlet total  10.52  sum N ln y =  -8.1981    difference +0.00e+00
  perturbed (7.56)         inlet total  10.56  sum N ln y =  -8.2115
                           outlet total  10.56  sum N ln y =  -8.2115    difference +0.00e+00


**The cancellation is exact, and it is exact for a reason that has nothing to do with the
numbers.** Look at the two mole-number sets:

$$\text{in: }\{1,\;2,\;7.52\}\qquad\qquad\text{out: }\{1,\;2,\;7.52\}$$

CH$_4$ + 2O$_2$ $\to$ CO$_2$ + 2H$_2$O has $\sum_i\nu_i = 0$, so the total is unchanged; and with
*stoichiometric* air the products inherit the reactants' mole numbers exactly, one for one. So
$\sum_i N_i\ln y_i$ is the same sum with its terms relabeled. Change either condition -- excess air,
or a reaction with $\sum\nu_i \neq 0$ -- and it breaks. That is what Problems 14.23 to 14.27 are
about, and it is what the next illustration runs into.

In [5]:
# Excess air breaks it. By how much is a number, not an adjective.
rows = []
for excess in (0.0, 0.2, 0.5, 1.0):
    O2_in = 2.0 * (1 + excess)
    inlet = {"CH4": 1.0, "O2": O2_in, "N2": O2_in * AIR}
    outlet = {"CO2": 1.0, "H2O": 2.0, "O2": O2_in - 2.0, "N2": O2_in * AIR}
    si, _ = N_ln_y(inlet)
    so, _ = N_ln_y(outlet)
    correction = R * T_AMB * (so - si) / 1e3
    rows.append({"excess air": f"{excess:.0%}", "sum N ln y in": si,
                 "sum N ln y out": so, "RT x difference (kJ)": correction,
                 "W_S,max (kJ/mol)": dG_rxn + correction,
                 "error if dropped": abs(correction / (dG_rxn + correction))})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

excess air  sum N ln y in  sum N ln y out  RT x difference (kJ)  W_S,max (kJ/mol)  error if dropped
        0%        -8.1981         -8.1981                0.0000         -801.1000            0.0000
       20%        -9.3510        -10.4323               -2.6805         -803.7805            0.0033
       50%       -11.0340        -12.9435               -4.7334         -805.8334            0.0059
      100%       -13.7602        -16.5328               -6.8727         -807.9727            0.0085


## Illustration 14.5-2 -- *n*-octane, a liquid fuel

$$\mathrm{C_8H_{18}}+12.5\,\mathrm{O_2}+47\,\mathrm{N_2}\rightarrow8\,\mathrm{CO_2}+9\,\mathrm{H_2O}+47\,\mathrm{N_2}$$

Two things change from the methane case. The octane is a **liquid** at ambient conditions, so it is
not in the gas mixture at all and contributes no $\ln y$ term; and $\sum_i\nu_i \neq 0$, so nothing
cancels.

 Appendix A.IV's entry is `nC8H18(l)`, $\Delta_fG^{\circ} = +7.4$ kJ/mol -- the **liquid**. The
gas-phase value is about $+16.4$, and this illustration says twice that the octane is a liquid, so
the liquid value is the one its own reasoning requires.

In [6]:
print(f"  dGf(nC8H18(l)) = {dGf('nC8H18(l)'):+.1f} kJ/mol   (Appendix A.IV, liquid)")
dG_oct = 8 * dGf("CO2") + 9 * dGf("H2O(g)") - dGf("nC8H18(l)") - 12.5 * dGf("O2")
print(f"        8 x dGf(CO2)     = {8*dGf('CO2'):+10.1f}")
print(f"        9 x dGf(H2O,g)   = {9*dGf('H2O(g)'):+10.1f}")
print(f"       -1 x dGf(C8H18,l) = {-dGf('nC8H18(l)'):+10.1f}")
print(f"    -12.5 x dGf(O2)      = {-12.5*dGf('O2'):+10.1f}   (zero by definition)")
print(f"\n  [a] W_S,max = dG_rxn = {dG_oct:+.1f} kJ/mol octane")
assert abs(dG_oct - (-5220.0)) < 0.05

  dGf(nC8H18(l)) = +7.4 kJ/mol   (Appendix A.IV, liquid)
        8 x dGf(CO2)     =    -3155.2
        9 x dGf(H2O,g)   =    -2057.4
       -1 x dGf(C8H18,l) =       -7.4
    -12.5 x dGf(O2)      =       -0.0   (zero by definition)

  [a] W_S,max = dG_rxn = -5220.0 kJ/mol octane


In [7]:
# Part [b]: the exact equation. Octane is liquid, so it is absent from the gas phase.
inlet = {"O2": 12.5, "N2": 12.5 * AIR}
outlet = {"CO2": 8.0, "H2O": 9.0, "N2": 12.5 * AIR}
si, ti = N_ln_y(inlet)
so, to = N_ln_y(outlet)

tab_in = pd.DataFrame([{"species": s, "moles": N, "y": N / ti,
                        "N ln y": N * np.log(N / ti)} for s, N in inlet.items()])
tab_out = pd.DataFrame([{"species": s, "moles": N, "y": N / to,
                         "N ln y": N * np.log(N / to)} for s, N in outlet.items()])
print("  INLET   (total %.1f mol of gas)" % ti)
print(tab_in.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"    sum N ln y = {si:.3f}")
print("\n  OUTLET  (total %.1f mol of gas)" % to)
print(tab_out.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"    sum N ln y = {so:.3f}")

assert (ti, to) == (59.5, 64.0)                  # 12.5 + 47, and 8 + 9 + 47
assert abs(si - (-30.587)) < 5e-3 and abs(so - (-48.801)) < 5e-3

  INLET   (total 59.5 mol of gas)
species   moles      y   N ln y
     O2 12.5000 0.2101 -19.5031
     N2 47.0000 0.7899 -11.0839
    sum N ln y = -30.587

  OUTLET  (total 64.0 mol of gas)
species   moles      y   N ln y
    CO2  8.0000 0.1250 -16.6355
    H2O  9.0000 0.1406 -17.6549
     N2 47.0000 0.7344 -14.5106
    sum N ln y = -48.801


**The two totals carry part [b].** 59.5 mol of gas in carrying two
species, 64.0 mol out carrying three. The stream leaves both larger and more finely mixed, and
neither of those is free.

In [8]:
correction = R * T_AMB * (so - si) / 1e3
print(f"  sum N ln y:  out {so:.3f}  -  in {si:.3f}  =  {so - si:+.3f}")
print(f"  RT x that  = {correction:+.2f} kJ/mol       -- NEGATIVE: more work available, not less")
print(f"\n  W_S,max = {dG_oct:+.1f} {correction:+.2f} = {dG_oct + correction:+.1f} kJ/mol octane")
print(f"  the correction is {abs(correction/(dG_oct+correction)):.2%} of the answer")

assert correction < 0                            # a source of availability, not a sink
assert abs(dG_oct + correction - (-5265.1)) < 0.1

  sum N ln y:  out -48.801  -  in -30.587  =  -18.214
  RT x that  = -45.15 kJ/mol       -- NEGATIVE: more work available, not less

  W_S,max = -5220.0 -45.15 = -5265.1 kJ/mol octane
  the correction is 0.86% of the answer


**Be precise about the direction.** The exact equation makes *more* work available
here, not less. The products are more finely mixed than the reactants -- 64 mol of gas carrying
three species where the inlet had 59.5 carrying two -- and mixing entropy is a **source** of
availability, not a sink. So the approximate equation *understates* the work by 0.86 %.

 That sign is the one to check by hand, because $\sum_i N_i\ln y_i$ is negative on both
sides and it is easy to difference the magnitudes instead of the sums. $|{-48.801}| - |{-30.587}|$
and $(-48.801) - (-30.587)$ differ in sign, and only the second is the quantity the balance
calls for.

## Illustration 14.5-3 -- glucose, and why 37 °C is different

$$\mathrm{C_6H_{12}O_6}+6\,\mathrm{O_2}\rightarrow6\,\mathrm{CO_2}+6\,\mathrm{H_2O}$$

At 25 °C the flow availability of each species *is* its Gibbs energy of formation, so part [b] is
one line. Part [c] is the interesting half: at 37 °C it is not, and $H$ and $S$ have to be carried
separately. Glucose is not in Appendix A.IV, so the illustration supplies its own data table.

In [9]:
# The illustration's own data table. Glucose is not in Appendix A.IV; water is the LIQUID.
data = pd.DataFrame([
    ("glucose", -1271.0, 209.2, 218.6),
    ("O2",          0.0,   0.0,  29.4),
    ("CO2",      -393.5,   3.0,  37.0),
    ("H2O",      -285.8, -163.3, 73.4),
], columns=["species", "dHf_kJ", "dSf_J_K", "Cp_J_K"]).set_index("species")

# part a: dGf = dHf - T dSf. The illustration asks the reader to complete the column.
data["dGf_kJ"] = data.dHf_kJ - T_AMB * data.dSf_J_K / 1e3
data["B_25_kJ"] = data.dGf_kJ                     # at T_amb, availability == dGf
print(data.to_string(float_format=lambda v: f"{v:.1f}"))

for s, g in (("glucose", -1333.4), ("O2", 0.0), ("CO2", -394.4), ("H2O", -237.1)):
    assert abs(data.loc[s, "dGf_kJ"] - g) < 0.05, s

         dHf_kJ  dSf_J_K  Cp_J_K  dGf_kJ  B_25_kJ
species                                          
glucose -1271.0    209.2   218.6 -1333.4  -1333.4
O2          0.0      0.0    29.4     0.0      0.0
CO2      -393.5      3.0    37.0  -394.4   -394.4
H2O      -285.8   -163.3    73.4  -237.1   -237.1


In [10]:
NU = {"glucose": -1.0, "O2": -6.0, "CO2": 6.0, "H2O": 6.0}
W_25 = sum(NU[s] * data.loc[s, "B_25_kJ"] for s in NU)
print(f"  [b] W_S,max(25 C) = {W_25:+.1f} kJ/mol glucose")
assert abs(W_25 - (-2455.7)) < 0.1

  [b] W_S,max(25 C) = -2455.7 kJ/mol glucose


In [11]:
# part c: correct H and S to body temperature with a constant Cp, then B = H - T S.
dT = T_BODY - T_AMB
ln_ratio = np.log(T_BODY / T_AMB)
print(f"  T_body - T_amb = {dT:.2f} K        ln(T_body/T_amb) = {ln_ratio:.5f}\n")

body = pd.DataFrame(index=data.index)
body["H_kJ"] = data.dHf_kJ + data.Cp_J_K * dT / 1e3
body["S_J_K"] = data.dSf_J_K + data.Cp_J_K * ln_ratio
body["B_kJ"] = body.H_kJ - T_BODY * body.S_J_K / 1e3
print(body.to_string(float_format=lambda v: f"{v:.3f}"))

W_37 = sum(NU[s] * body.loc[s, "B_kJ"] for s in NU)
print(f"\n  [c] W_S,max(37 C) = {W_37:+.1f} kJ/mol glucose")
print(f"      the 12 K costs {abs(W_37 - W_25):.1f} kJ/mol, "
      f"{abs((W_37-W_25)/W_25):.2%} of the total")

assert abs(W_37 - (-2441.7)) < 0.1
for s, H, Sv, B in (("glucose", -1268.4, 217.8, -1335.9), ("O2", 0.353, 1.160, -0.007),
                    ("CO2", -393.1, 4.460, -394.4), ("H2O", -284.9, -160.404, -235.2)):
    assert abs(body.loc[s, "H_kJ"] - H) < 0.1, s
    assert abs(body.loc[s, "S_J_K"] - Sv) < 0.05, s

  T_body - T_amb = 12.00 K        ln(T_body/T_amb) = 0.03946

             H_kJ    S_J_K      B_kJ
species                             
glucose -1268.377  217.826 -1335.935
O2          0.353    1.160    -0.007
CO2      -393.056    4.460  -394.439
H2O      -284.919 -160.404  -235.170

  [c] W_S,max(37 C) = -2441.7 kJ/mol glucose
      the 12 K costs 14.0 kJ/mol, 0.57% of the total


**Look again at the oxygen row, and not for its size.** At 37 °C
oxygen's availability is

$$B_{\rm O_2}=0.3528-310.15\times\frac{1.1601}{1000}=-0.0070\ \text{kJ/mol}$$

a difference of two numbers that agree to 2 %. So its **sign is not resolved at three decimals**:
carry $S$ as 1.12 instead of 1.160 and $B$ comes out $+0.005$ rather than $-0.007$. Six moles of it
move $W_{S,\max}$ by 0.10 kJ out of 2442 -- **0.004 %** -- so nothing downstream can detect the
difference.

**That is the trap, not the reassurance.** A quantity whose sign flips under rounding you cannot
see is exactly the kind that survives edition after edition. When a table entry is a small
difference of large numbers, either carry enough digits to fix its sign or print it as zero -- do
not print a signed value the arithmetic does not support.

In [12]:
# Efficiency, Eq. 14.5-8, and the numbers the section quotes.
print("  Eq. 14.5-8:  efficiency = electrical power generated / (-W_S,max)\n")
print(f"  methane, at the 35 % the section attributes to gas-fired plants:")
print(f"    delivered {0.35*abs(dG_rxn):.0f} of {abs(dG_rxn):.0f} kJ/mol; "
      f"{0.65*abs(dG_rxn):.0f} kJ/mol is lost")
print(f"  methane, at the 60 % the section attributes to co-generation:")
print(f"    delivered {0.60*abs(dG_rxn):.0f} kJ/mol\n")
print(f"  glucose, at the 32 % the section attributes to the ADP-to-ATP step:")
print(f"    {0.32*abs(W_37):.0f} kJ/mol glucose stored as ATP")
print(f"    at ~30.5 kJ per mole of ATP that is {0.32*abs(W_37)/30.5:.1f} ATP per glucose")

  Eq. 14.5-8:  efficiency = electrical power generated / (-W_S,max)

  methane, at the 35 % the section attributes to gas-fired plants:
    delivered 280 of 801 kJ/mol; 521 kJ/mol is lost
  methane, at the 60 % the section attributes to co-generation:
    delivered 481 kJ/mol

  glucose, at the 32 % the section attributes to the ADP-to-ATP step:
    781 kJ/mol glucose stored as ATP
    at ~30.5 kJ per mole of ATP that is 25.6 ATP per glucose


## Your turn

1. The $RT\ln y$ terms cancel exactly for methane in stoichiometric air. Prove it in general: show
   that they cancel whenever $\sum_i\nu_i=0$ **and** the product mole numbers are a permutation of
   the reactant ones. Then find another combustion reaction in Appendix A.IV with that property, or
   argue that methane is the only one.
2. The cancellation in Illustration 14.5-1 survives a nitrogen number that is slightly wrong -- the
   cell above shows it holds for 7.52 and for 7.56 alike. Find a quantity in the same illustration
   that a 0.04 mol error in nitrogen *would* change, and by how much. What does that say about
   which intermediates are safe to round before printing?
3. Rework Illustration 14.5-2 with the water leaving as a **liquid** rather than a vapor. At 25 °C
   and 1 bar, 9 mol of water in 64 mol of gas is 14 mol %, against a saturation value near 3 % --
   so most of it must condense. How much does that change $W_{S,\max}$, and which of the two answers
   should the illustration be reporting?
4. The exact availability equation makes 0.86 % *more* work available from octane, not less. Work
   the sign out from first principles: is mixing the reactants and mixing the products a gain or a
   loss of availability, and does the answer depend on which side has more moles of gas? Then show
   what you get if you difference $|\sum N\ln y|$ instead of $\sum N\ln y$, and say why that is the
   easy mistake to make.
5. Glucose's availability changes by only 14 kJ/mol between 25 °C and 37 °C, out of 2456. Show why
   the change is so small -- the $\Delta H$ and $T\Delta S$ corrections nearly cancel -- and find the
   temperature at which the availability of this reaction would be 5 % below its 25 °C value. Is that
   temperature physically reachable for the organism?
6. Oxygen's availability at 37 °C is $-0.007$ kJ/mol, a difference of two numbers that agree to 2 %.
   Work out how many digits of $C_P$ you need to fix its sign, and find one other entry in these
   three illustrations that is a small difference of large numbers. Which quantities in a thermodynamic table need extra digits, and which do not?